Stage 1

Pipeline to extract information from Glassdoor - Reviews oof Microsoft Company

In [ ]:
import asyncio
import os
import random
import pandas as pd
from playwright.async_api import async_playwright, Playwright

async def scrape_microsoft_reviews(playwright: Playwright):
    # Configuración del perfil de Edge
    user_data_dir = os.path.join(os.environ['LOCALAPPDATA'], r'Microsoft\Edge\HP')
    
    context = await playwright.chromium.launch_persistent_context(
        user_data_dir,
        channel="msedge",
        headless=False,
        args=["--disable-blink-features=AutomationControlled"],
        ignore_default_args=["--enable-automation"]
    )

    page = context.pages[0] if context.pages else await context.new_page()
    url = "https://www.glassdoor.com.mx/Opiniones/Microsoft-Opiniones-E1651.htm"
    
    reviews_list = []

    try:
        print(f"Navegando a: {url}...")
        # CAMBIO 1: Usamos domcontentloaded y un timeout más largo para evitar el error de los 30s
        await page.goto(url, wait_until="domcontentloaded", timeout=60000)

        # Pausa para simular comportamiento humano
        await asyncio.sleep(random.uniform(4, 7))

        print("Si aparece un captcha, resuélvelo manualmente en la ventana del navegador...")
        
        # Esperamos a que el contenedor principal sea visible
        try:
            await page.wait_for_selector('[data-test="review-detail"]', timeout=30000)
        except Exception:
            print("Error: No se detectaron las reseñas. Revisa si el Captcha bloqueó el acceso.")
            return

        # Localizamos todos los contenedores de reseñas
        reviews = await page.locator('[data-test="review-detail"]').all()
        print(f"Se encontraron {len(reviews)} reseñas para procesar.")

        for i, review in enumerate(reviews):
            try:
                # Pausa entre cada reseña para no saturar
                await asyncio.sleep(random.uniform(2, 4))
                
                # CAMBIO 2: Selectores de clase corregidos (unidos con puntos)
                # Nota: Se usa .first y timeouts cortos (2s) para agilizar si el dato no existe
                
                titulo_loc = review.locator('.heading_Heading__aomVx.heading_Level3__py3_P')
                rating_loc = review.locator('.ReviewRating_ratingLabel__uZhnh')
                puesto_loc = review.locator('.ContentAvatarTags_avatarLabel__Nb7Nh')
                estatus_loc = review.locator('.text-with-icon_TextWithIcon__Fs1BS')
                pros_loc = review.locator('[data-test="review-text-PROS"]')
                cons_loc = review.locator('[data-test="review-text-CONS"]')

                # Extracción segura de texto
                titulo = await titulo_loc.inner_text(timeout=2000) if await titulo_loc.count() > 0 else "N/A"
                rating = await rating_loc.inner_text(timeout=2000) if await rating_loc.count() > 0 else "N/A"
                puesto = await puesto_loc.inner_text(timeout=2000) if await puesto_loc.count() > 0 else "N/A"
                estatus = await estatus_loc.first.inner_text(timeout=2000) if await estatus_loc.count() > 0 else "N/A"
                ventajas = await pros_loc.inner_text(timeout=2000) if await pros_loc.count() > 0 else "N/A"
                desventajas = await cons_loc.inner_text(timeout=2000) if await cons_loc.count() > 0 else "N/A"

                reviews_list.append({
                    "Titulo": titulo.strip(),
                    "Rating": rating.strip(),
                    "Puesto": puesto.strip(),
                    "Estatus de empleado": estatus.strip(),
                    "Ventajas": ventajas.strip(),
                    "Desventajas": desventajas.strip()
                })
                print(f" [{i+1}/{len(reviews)}] Extraída: {titulo[:30]}...")

            except Exception as e:
                print(f" Error omitido en reseña {i+1}: {e}")
                continue

        # Generación del archivo CSV
        if reviews_list:
            df = pd.DataFrame(reviews_list)
            df.to_csv("reviews_microsoft.csv", index=False, encoding='utf-8-sig')
            print(f"\n ¡Éxito! Archivo 'microsoft_resenas.csv' creado con {len(df)} filas.")
        else:
            print("\n No se pudo extraer ninguna reseña. Verifica los selectores de clase.")

    finally:
        await asyncio.sleep(2)
        await context.close()

async def main():
    async with async_playwright() as playwright:
        await scrape_microsoft_reviews(playwright)

if __name__ == "__main__":
    asyncio.run(main())

Stage 2 

Model to classifier and make a sentiment analysis

Stage 3

Create a Pipeline to MLOps